In [15]:
import geopandas as gpd
import matplotlib.pyplot as plt
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry


# Loading and looking at shapefiles

In [8]:
# Load shapefiles
gdf_regions = gpd.read_file("../data/shapefiles/nve_region/NVE_Region.shp")
gdf_zones = gpd.read_file("../data/shapefiles/bidding_zones/biddingzones.shp")

print("NVE regions:")
print(gdf_regions.head())
print(gdf_regions.crs)

print("\nBidding zones:")
print(gdf_zones.head())
print(gdf_zones.crs)

NVE regions:
   OBJECTID REGION         REGIONNAVN POSTNUMMER   POSTSTED  \
0       1.0     RV        Region Vest       6801     FÃ¸rde   
1       2.0     RM  Region Midt-Norge       7030  Trondheim   
2       3.0     RN        Region Nord       8514     Narvik   
3       4.0    RÃ        Region Ãst       2307      Hamar   
4       5.0     RS        Region SÃ¸r       3103  TÃ¸nsberg   

               BESÃKSADR            ADRESSE  \
0       Naustdalsveien 1B        Postboks 53   
1  Abelsgate 9, Trondheim        Abelsgate 9   
2       Kongensgate 52-54  Kongensgate 52-54   
3           Vangsvegen 73      Postboks 4223   
4   Anton Jenssens gate 7      Postboks 2124   

                                            geometry  
0  POLYGON ((193105.76 6929797.839, 191705.75 692...  
1  POLYGON ((391250 7388620, 392270 7386790, 3937...  
2  POLYGON ((1109096.413 7844647.502, 1105115.343...  
3  POLYGON ((253748.22 6678130.91, 253645.58 6678...  
4  POLYGON ((253748.22 6678130.91, 253917.63

# Generate random coordinates

In [26]:
# Number of random points per region
K = 1000

# NVE regions
points_nve = gdf_regions["geometry"].sample_points(
    size=K, random_state=42
)

# Electricity bidding zones
points_zones = gdf_zones["geometry"].sample_points(
    size=K, random_state=42
)

points_nve_wgs84 = points_nve.to_crs(epsg=4326)
points_zones_wgs84 = points_zones.to_crs(epsg=4326)

# Store coordinates by region
nve_coordinates = {}
zone_coordinates = {}

for idx, multipoint in points_nve_wgs84.items():
    region = gdf_regions.loc[idx, "REGION"]

    nve_coordinates[region] = []

    for point in multipoint.geoms:
        lat = point.y
        lon = point.x

        nve_coordinates[region].append((lon, lat))


for idx, multipoint in points_zones_wgs84.items():
    zone = gdf_zones.loc[idx, "budomr"]

    zone_coordinates[zone] = []

    for point in multipoint.geoms:
        lat = point.y
        lon = point.x

        zone_coordinates[zone].append((lon, lat))

print("NVE regions:")
for region, coordinates in nve_coordinates.items():
    print(region, len(coordinates), coordinates[:2])

print("\nBidding zones:")
for zone, coordinates in zone_coordinates.items():
    print(zone, len(coordinates), coordinates[:2])

print("NO3 coordinates:", len(zone_coordinates["NO 3"]))
print("First coordinate:", zone_coordinates["NO 3"][0])

NVE regions:
RV 1000 [(5.03284734873187, 59.86736213243584), (7.1606994221273945, 59.935332910872226)]
RM 1000 [(14.351798422027803, 66.22904759712564), (13.514472517440893, 64.37792251858549)]
RN 1000 [(17.439677010757798, 68.61158040900165), (19.35770755654209, 69.81238895581274)]
RÃ 1000 [(11.113662887069225, 60.58739371856167), (10.442966050117843, 60.972129798666714)]
RS 1000 [(8.18652838647737, 59.44900753652939), (4.57465730568268, 59.202169924370885)]

Bidding zones:
NO 1 1000 [(10.69253137382541, 60.155719883388535), (10.41739538068675, 59.879018950340566)]
NO 2 1000 [(8.739248674148863, 59.87260665799384), (7.580967269648451, 58.119369447737455)]
NO 3 1000 [(9.099069341615477, 63.58998956374261), (11.65164165113133, 64.5841778154394)]
NO 4 1000 [(20.010391573690924, 68.89959411064771), (24.2541867484763, 69.05571679263937)]
NO 5 1000 [(7.275323953692646, 60.24732313759013), (7.443821929006578, 60.18001375087683)]
NO3 coordinates: 1000
First coordinate: (9.099069341615477, 63

# Set up Open-Meteo API Client

In [24]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Select one random coordinate from NO3
lon, lat = zone_coordinates["NO 3"][0]

# Historical Weather API using ERA5 reanalysis
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2024-01-01",
    "end_date": "2024-01-07",
    "hourly": ["temperature_2m", "precipitation"],
    "models": "era5",
    "timezone": "GMT"
}

responses = openmeteo.weather_api(url, params=params)

# Process first location
response = responses[0]

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data
# Variable order must match the order requested in params
hourly = response.Hourly()

hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation

# Convert to Pandas DataFrame
df_weather = pd.DataFrame(data=hourly_data)

print("\nHourly data:")
display(df_weather.head())

# Check dimensions, missing values and descriptive statistics
print("\nShape:", df_weather.shape)
print("\nMissing values:")
print(df_weather.isna().sum())

display(df_weather.describe())

OpenMeteoRequestsError: failed to request 'https://archive-api.open-meteo.com/v1/archive': 503 Server Error: Service Unavailable for url: https://archive-api.open-meteo.com/v1/archive?latitude=61.74343508046112&longitude=4.3797445483425586&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m&hourly=precipitation&models=era5&timezone=GMT&format=flatbuffers

# Creating reusable function
Making the api call a function which takes regional identifier and time interval as input. Also includes batching to reduce the HTTP overhead.

In [ ]:
from pathlib import Path

def download_weather(region, start_date, end_date):
    """
    Download hourly ERA5 weather data for all sampled coordinates
    in a region, saving each completed batch as a CSV file.
    """

    # Look up sampled coordinates for the selected region
    coordinates = zone_coordinates[region]
    url = "https://archive-api.open-meteo.com/v1/archive"

    batch_size = 10
    weather_data = []

    # Create folder for the selected region and time interval
    folder = Path("data/weather") / f"{region.replace(' ', '')}_{start_date}_{end_date}"
    folder.mkdir(parents=True, exist_ok=True)

    # Loop through coordinates in batches of 10
    for i in range(0, len(coordinates), batch_size):

        batch = coordinates[i:i + batch_size]
        batch_file = folder / f"batch_{i:04d}.csv"

        # Load previously downloaded batches
        if batch_file.exists():
            print(f"Loading saved batch {i}")
            df = pd.read_csv(batch_file, parse_dates=["date"])
            weather_data.append(df)
            continue

        # Extract latitude and longitude lists
        longitudes = [lon for lon, lat in batch]
        latitudes = [lat for lon, lat in batch]

        params = {
            "latitude": latitudes,
            "longitude": longitudes,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ["temperature_2m", "precipitation"],
            "models": "era5",
            "timezone": "GMT"
        }

        # Download weather data for the batch
        responses = openmeteo.weather_api(url, params=params)

        batch_data = []

        # Process each location in the batch
        for j, response in enumerate(responses):

            hourly = response.Hourly()

            temperature = hourly.Variables(0).ValuesAsNumpy()
            precipitation = hourly.Variables(1).ValuesAsNumpy()

            # Construct DataFrame
            df = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=hourly.Interval()),
                    inclusive="left"
                ),
                "temperature_2m": temperature,
                "precipitation": precipitation
            })

            # Add location information
            lon, lat = batch[j]

            df["region"] = region
            df["location_id"] = i + j
            df["latitude"] = lat
            df["longitude"] = lon

            batch_data.append(df)

        # Combine data from the current batch
        df_batch = pd.concat(batch_data, ignore_index=True)

        # Save completed batch as CSV
        df_batch.to_csv(batch_file, index=False)

        weather_data.append(df_batch)

        print(
            f"Saved {min(i + batch_size, len(coordinates))} "
            f"/ {len(coordinates)} locations"
        )

    # Combine all batches into one DataFrame
    return pd.concat(weather_data, ignore_index=True)

In [ ]:
df_weather = download_weather(
    "NO 3",
    "2024-01-01",
    "2024-12-31"
)

# AI Usage

# Log